In WHERE, returning one value (a “scalar” subquery):
SELECT post_id, likes
FROM posts
WHERE likes > (SELECT AVG(likes) FROM posts)

In FROM, returning a whole table (a “derived table”), which must have an alias:
SELECT COUNT(*)
FROM (SELECT team FROM toy GROUP BY team HAVING COUNT(*) > 2) AS big_teams

Also in SELECT: a scalar subquery gives you “% of total” in one line:
SELECT platform, COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM posts), 1) AS pct
FROM posts
GROUP BY platform

CTEs: subqueries with names

A CTE (Common Table Expression) is a subquery you name at the top, with WITH. It’s the same idea as (c), but far easier to read:

WITH brand_avg AS (
    SELECT brand, AVG(likes) AS avg_likes
    FROM posts
    GROUP BY brand
)
SELECT *
FROM brand_avg
WHERE avg_likes > 1500

WITH a AS (...),
     b AS (SELECT ... FROM a)
SELECT ... FROM b

A CTE only exists for that one query. You can’t use it in the next cell.

CTE or subquery? They do the same job.

For a one-line value like the overall average, a subquery is fine.
For anything with more than one step, use a CTE. Interviewers prefer it because it’s readable.

The big one: aggregate first, then join

This is the proper fix for row explosion (3.3, D4 and P13). Squash each table down to one row per key before joining.

-- ✗ explodes: team x's total becomes 26, not 13
SELECT t.team, SUM(t.score) FROM toy t JOIN sponsors s ON t.team = s.team GROUP BY t.team

-- ✓ one row per team on each side, then join
WITH team_scores AS (
    SELECT team, SUM(score) AS total FROM toy GROUP BY team
),
team_sponsors AS (
    SELECT team, COUNT(*) AS n_sponsors FROM sponsors GROUP BY team
)
SELECT ts.team, ts.total, sp.n_sponsors
FROM team_scores ts
LEFT JOIN team_sponsors sp ON ts.team = sp.team

WITH brand_avg AS (
    SELECT brand, AVG(likes) AS avg_likes FROM posts GROUP BY brand
)
SELECT p.post_id, p.brand, p.likes, ba.avg_likes
FROM posts p
JOIN brand_avg ba ON p.brand = ba.brand
WHERE p.likes > ba.avg_likes

-- employees who manage nobody
SELECT name FROM staff
WHERE emp_id NOT IN (SELECT manager_id FROM staff)
This returns nothing. manager_id contains a NULL (Ana’s), and x NOT IN (1, 2, 9, NULL) can never be true. SQL can’t rule out that the unknown NULL equals x.
    Add WHERE manager_id IS NOT NULL inside the subquery.
Or use the LEFT JOIN anti-join from 3.3, which is NULL-safe. This is the safer habit.

Watch out: a “one value” subquery that returns many

WHERE likes > (SELECT AVG(likes) FROM posts GROUP BY brand) returns 5 rows.

PostgreSQL and most other databases raise an error.
SQLite silently uses the first row. You get a number, but it’s wrong.

SUBQUERIES
WHERE x > (SELECT AVG(x) FROM t)            scalar: must return 1 row, 1 col
WHERE k IN (SELECT k FROM t2 WHERE ...)     list
FROM (SELECT ... GROUP BY ...) AS sub       derived table, needs an alias
SELECT ..., 100.0*COUNT(*)/(SELECT COUNT(*) FROM t)   % of total

CTEs
WITH a AS (SELECT ...),
     b AS (SELECT ... FROM a)               one WITH, commas between
SELECT ... FROM b                           lives for this query only

PATTERNS
- Aggregate first, then join → one row per key on each side → no explosion
- Row vs its own group: CTE of group stats, JOIN back on the key, WHERE row > stat
- Top per group: CTE of MAX per group, JOIN back on key AND value
- An alias made inside a CTE can be used in WHERE outside it

TRAPS
- NOT IN + a NULL in the list → returns nothing. Use the anti-join or filter NULLs.
- Scalar subquery that returns many rows → SQLite silently takes the first. Run the inner query alone.
- Build inside-out: run the inner query alone first.

In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import sqlite3
con = sqlite3.connect("../data/social.db")
tables = {"posts": "../data/social_posts.csv", "brands": "../data/brands.csv",
          "orders": "../data/gate_orders.csv", "customers": "../data/gate_customers.csv"}
for name, path in tables.items():
    pd.read_csv(path).to_sql(name, con, index=False, if_exists="replace")

toys = {
    "toy": pd.DataFrame({"name": ["a","b","c","d","e"], "team": ["x","y","x","y","x"],
                         "score": [5, 12, None, 20, 8], "city": ["UK","US","UK",None,"US"]}),
    "teams": pd.DataFrame({"team": ["x","y","z"], "coach": ["Kim","Lee","Ray"]}),
    "sponsors": pd.DataFrame({"team": ["x","x","y"], "sponsor": ["Nike","Puma","Adidas"]}),
    "staff": pd.DataFrame({"emp_id": [1,2,3,4,5], "name": ["Ana","Ben","Cat","Dev","Eli"],
                           "manager_id": [None, 1, 1, 2, 9]}),
}
for name, df in toys.items():
    df.to_sql(name, con, index=False, if_exists="replace")

def q(sql):
    return pd.read_sql(sql, con)

In [2]:
q("SELECT * FROM toy")
q("SELECT name, score FROM toy WHERE score > (SELECT AVG(score) FROM toy)")
# 11.25, b, d
#one val subquery

,name,team,score,city
0,a,x,5.0,UK
1,b,y,12.0,US
2,c,x,NaN,UK
3,d,y,20.0,NaN
4,e,x,8.0,US


,name,score
0,b,12.0
1,d,20.0


In [3]:
q("SELECT name FROM toy WHERE team IN (SELECT team FROM sponsors WHERE sponsor = 'Nike')")
q("SELECT * FROM sponsors")
#x. a,c,e 
#list subquery

,name
0,a
1,c
2,e


,team,sponsor
0,x,Nike
1,x,Puma
2,y,Adidas


In [4]:
q("SELECT name FROM staff WHERE emp_id NOT IN (SELECT manager_id FROM staff WHERE manager_id IS NOT NULL)")
q("SELECT * FROM staff")
#cat, dev, eli
#This returns nothing. manager_id contains a NULL (Ana’s), and x NOT IN (1, 2, 9, NULL) can never be true. SQL can’t rule out that the unknown NULL equals x.

,name
0,Cat
1,Dev
2,Eli


,emp_id,name,manager_id
0,1,Ana,NaN
1,2,Ben,1.0
2,3,Cat,1.0
3,4,Dev,2.0
4,5,Eli,9.0


In [5]:
# -- ✗ explodes: team x's total becomes 26, not 13
q("""SELECT t.team, SUM(t.score) 
FROM toy t 
JOIN sponsors s 
ON t.team = s.team 
GROUP BY t.team""")
# x has 2 sponsors so its result doubles due to the match returning 2 rows

# -- ✓ one row per team on each side, then join
q("""WITH team_scores AS (
    SELECT team, SUM(score) AS total FROM toy GROUP BY team
),
team_sponsors AS (
    SELECT team, COUNT(*) AS n_sponsors FROM sponsors GROUP BY team
)
SELECT ts.team, ts.total, sp.n_sponsors
FROM team_scores ts
LEFT JOIN team_sponsors sp ON ts.team = sp.team""")
#correct answer because it creates a new col of score total, and sponsor total and then joins it on the team key 

,team,SUM(t.score)
0,x,26.0
1,y,32.0


,team,total,n_sponsors
0,x,13.0,2
1,y,32.0,1


In [6]:
q("""WITH tm AS (SELECT team, AVG(score) AS a FROM toy GROUP BY team)
SELECT t.name, t.score, tm.a
FROM toy t JOIN tm ON t.team = tm.team
WHERE t.score > tm.a""")

q("""SELECT team, AVG(score) AS a FROM toy GROUP BY team""")
#x-6.5, y-16.0
# d, e

,name,score,a
0,d,20.0,16.0
1,e,8.0,6.5


,team,a
0,x,6.5
1,y,16.0


In [7]:
q("""SELECT COUNT(*) FROM (SELECT team FROM toy GROUP BY team HAVING COUNT(*) > 2) AS big""")
#x, 1--just one team (x) with count>2
q("""SELECT team FROM toy GROUP BY team HAVING COUNT(*) > 2""")

,COUNT(*)
0,1


,team
0,x


In [8]:
# B1. Above-average posts

# (a) Find the overall average likes (1 dp).
# (b) Count the posts above it, using a subquery.

q("SELECT COUNT(*) FROM posts WHERE likes>(SELECT ROUND(AVG(likes),1) FROM posts)")

,COUNT(*)
0,103


In [9]:
# B2. IN / NOT IN

# (a) Using IN and a subquery, count the posts from UK-based brands.
q("SELECT COUNT(*) FROM posts WHERE brand IN (SELECT brand FROM brands WHERE hq_country = 'UK')")
# (b) Does this match P7 from the practice set?
#yes
# (c) Using NOT IN, count the posts whose brand isn’t in brands at all.
q("SELECT COUNT(*) FROM posts WHERE brand NOT IN (SELECT brand FROM brands)")
# (d) Is NOT IN safe here? Check whether brands.brand has any NULLs.
q("SELECT brand FROM brands WHERE brand IS NULL")
#yeah, no nulls so we can use it here

,COUNT(*)
0,118


,COUNT(*)
0,66


,brand


In [10]:
# B3. Share of posts

# (a) For each platform, show the post count and its % of all posts (1 dp), using a scalar subquery in SELECT.
q("""SELECT platform, COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM posts), 1) AS pct
FROM posts
GROUP BY platform""")
# (b) Do the % values add up to 100?
#yeah

,platform,n,pct
0,Facebook,64,21.3
1,Instagram,132,44.0
2,TikTok,104,34.7


In [11]:
# B4. Aggregate first, then join

# (a) Write a CTE brand_likes with one row per brand: brand, total_likes.
q("""WITH brand_likes AS(
    SELECT brand, SUM(likes) AS total_likes
    FROM posts
GROUP BY brand
)

SELECT b.brand, b.monthly_budget, bl.total_likes, ROUND(total_likes*1.0/monthly_budget*1.0, 2) AS likes_per_pound
FROM brands b
LEFT JOIN brand_likes bl
ON bl.brand=b.brand
ORDER BY likes_per_pound DESC
""")
    
# (b) LEFT JOIN brands to it and show brand, monthly_budget, total_likes and likes per £ (2 dp).
# (c) Does it match P9 from the practice set?

,brand,monthly_budget,total_likes,likes_per_pound
0,Glowly,12000,101657.0,8.47
1,FitFuel,9500,66044.0,6.95
2,SnackBox,25000,161327.0,6.45
3,UrbanThread,8000,47147.0,5.89
4,PetPal,3000,NaN,NaN


In [12]:
# B5. Beat your own brand

# (a) Using a CTE of brand averages, count all posts that beat their own brand’s average.
q("""
WITH brand_avgs AS (
    SELECT 
        brand, 
        AVG(likes) AS avg_likes 
    FROM posts 
    GROUP BY brand 
)
SELECT p.brand, COUNT(*) AS total_above_average_posts
FROM posts p
JOIN brand_avgs ba
    ON p.brand = ba.brand
WHERE p.likes > ba.avg_likes
GROUP BY p.brand
ORDER BY total_above_average_posts DESC
""")
# (b) Break that count down per brand, highest first.
# (c) Does HomeNest appear? Why can it appear here when it disappeared in so many 3.3 queries?
# we made brand avgs using posts where homenest exists, but when we kept the left table as brands, it disappoeared because it isnt in brands

,brand,total_above_average_posts
0,Glowly,22
1,UrbanThread,21
2,FitFuel,21
3,SnackBox,20
4,HomeNest,20


In [13]:
# B6. Orders per customer

# (a) Write a CTE cust_orders with customer_id, n_orders, using orders only.
# Query 1: Tier Analysis
q("""
WITH cust_orders AS (
    SELECT customer_id, COUNT(order_id) AS n_orders
    FROM orders
    GROUP BY customer_id
)
SELECT c.tier, ROUND(AVG(co.n_orders), 2) AS avg_orders, ROUND(MAX(co.n_orders), 2) AS max_orders
FROM cust_orders co
JOIN customers c ON co.customer_id = c.customer_id
GROUP BY c.tier;
""")

# Query 2: Above Average Count
q("""
WITH cust_orders AS (
    SELECT customer_id, COUNT(order_id) AS n_orders
    FROM orders
    GROUP BY customer_id
)
SELECT COUNT(*) AS above_avg_customer_count
FROM cust_orders
WHERE n_orders > (SELECT AVG(n_orders) FROM cust_orders);
""")


# (b) From that CTE, find the average and the maximum orders per customer (2 dp).
# (c) Join it to customers and show average orders per customer by tier (2 dp).
# (d) How many customers have more orders than the average? (Use the CTE twice: once in FROM and once in a subquery.)

,tier,avg_orders,max_orders
0,basic,3.29,10.0
1,plus,3.55,8.0
2,premium,3.87,8.0


,above_avg_customer_count
0,59


In [14]:
# B7. Customer segments

# (a) Using cust_orders, label each customer with CASE: 'heavy' for 6+ orders, 'regular' for 3–5, 'light' for 1–2.

q("""
WITH cust_orders AS (
    SELECT customer_id, COUNT(order_id) AS n_orders
    FROM orders
    GROUP BY customer_id
)

SELECT COUNT(*) AS customers, 
CASE 
WHEN n_orders<=2 THEN 'light'
WHEN n_orders<=5 THEN 'regular'
ELSE 'heavy' END AS segment
FROM cust_orders
GROUP BY segment""")

# (b) Count customers per segment.
# (c) Sanity check: do the segments add up to the number of rows in cust_orders?
#yes

,customers,segment
0,10,heavy
1,35,light
2,76,regular


In [15]:
# B8. Top post per brand

# (a) Write a CTE with each brand’s MAX(likes).

q("""WITH brand_max AS(
    SELECT brand, MAX(likes) AS max
    FROM posts
GROUP BY brand
)

SELECT bm.max, p.post_id, p.brand
FROM posts p
JOIN brand_max bm
ON bm.brand= p.brand
AND bm.max= p.likes
ORDER BY bm.max DESC
""")

# (b) Join it back to posts to get the post_id of each brand’s top post. (Hint: join on brand and likes.)
# (c) Which brand owns the single most-liked post overall?
# Snackbox

,max,post_id,brand
0,13577,P0019,SnackBox
1,5761,P0210,Glowly
2,4970,P0190,FitFuel
3,3384,P0271,UrbanThread
4,2032,P0180,HomeNest


In [16]:
# B9. Revenue per customer, by country (two steps)

# (a) In a CTE, calculate delivered revenue per customer (use the REPLACE/CAST trick from 3.3).
q("""WITH delivered_rev AS(
    SELECT customer_id, SUM(CAST(REPLACE(unit_price, '$', '') AS DECIMAL (10,2)) * quantity) AS revenue
    FROM orders
    WHERE status='delivered'
GROUP BY customer_id )

SELECT c.
country, ROUND(AVG(dr.revenue),2) AS avg_rev, COUNT(c.customer_id) AS cust_per_country
FROM delivered_rev dr
JOIN customers c
ON c.customer_id= dr.customer_id
GROUP BY c.country
ORDER BY avg_rev DESC
""")
# (b) Join it to customers and show the average revenue per customer by country (2 dp), highest first, plus how many customers are in each.
# (c) Which country’s customers spend the most on average?
#FR

,country,avg_rev,cust_per_country
0,FR,462.03,18
1,DE,435.81,12
2,UK,362.13,47
3,US,287.67,37


In [17]:
# B10. Fix P13 properly

# (a) Using a CTE that aggregates posts to one row per brand first, show each category’s total budget and total posts, with no inflation.
q("""WITH agg AS(
    SELECT brand, COUNT(*) AS total_posts
    FROM posts 
    GROUP BY brand)

SELECT b.category, SUM(b.monthly_budget), SUM(a.total_posts)
FROM brands b
LEFT JOIN agg a
ON b.brand=a.brand
GROUP BY b.category
""")     
# (b) Check food: the budget must be 25,000 and there should be 60 posts.

,category,SUM(b.monthly_budget),SUM(a.total_posts)
0,beauty,12000,62.0
1,fashion,8000,56.0
2,fitness,9500,56.0
3,food,25000,60.0
4,pets,3000,NaN


In [18]:
q("""SELECT e.name
FROM staff e                        
LEFT JOIN staff r                    
  ON r.manager_id = e.emp_id
WHERE r.manager_id IS NULL""") 


,name
0,Cat
1,Dev
2,Eli


In [19]:
q("""SELECT brand, ROUND(AVG(likes), 1) AS avg_l
FROM posts
GROUP BY brand
HAVING avg_l > 1500""")

q("""WITH avg_likes AS(
SELECT brand, ROUND(AVG(likes), 1) AS avg_l
FROM posts
GROUP BY brand)

SELECT brand, avg_l
FROM avg_likes
WHERE avg_l > 1500""")

,brand,avg_l
0,Glowly,1639.6
1,SnackBox,2688.8


,brand,avg_l
0,Glowly,1639.6
1,SnackBox,2688.8


In [20]:
q("""SELECT COUNT(*) FROM posts
WHERE likes > (SELECT AVG(likes) FROM posts)""")
# the inner query returns 5 rows, and SQLite silently used FitFuel’s average (1179), the first row.

,COUNT(*)
0,103


In [21]:
q("""
WITH team_scores AS 
(SELECT team, SUM(score) AS total 
FROM toy 
GROUP BY team),

team_sponsors AS 
(SELECT team, COUNT(*) AS n 
FROM sponsors 
GROUP BY team)

SELECT * 
FROM team_scores ts 
JOIN team_sponsors sp 
ON ts.team = sp.team

""")
# you only write WITH once, with commas between the CTEs.

,team,total,team,n
0,x,13.0,x,2
1,y,32.0,y,1
